# Imports

In [ ]:
import json
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import networkx as nx
import torch
import torch.nn.functional as F
from neo4j import GraphDatabase
import os
from torch_geometric.data import HeteroData
from torch_geometric.nn import GCNConv, SAGEConv
from torch_geometric.transforms import ToUndirected
from collections import Counter
from sklearn.metrics import f1_score, classification_report, confusion_matrix, accuracy_score


In [ ]:
with open(r"constitution_kg.json", "r", encoding="utf-8") as f:
    data = json.load(f)


# CATEGORY MAPPING


In [ ]:

CATEGORY_MAP = ["Rights", "Governance", "Judiciary", "Federalism", "Other"]

def map_category(article_number):
    """
    Map articles to simplified 4 categories.
    Safely extract numeric part from article_number string.
    """
    try:
        # Extract digits from the string
        n = int(''.join(filter(str.isdigit, str(article_number))))
    except ValueError:
        return "Other"  # fallback if conversion fails

    if 16 <= n < 47:
        return "Rights"
    elif 47 <= n < 101:
        return "Governance"
    elif 126 <= n < 142:
        return "Judiciary"
    elif (5 <= n < 16) or (142 <= n < 200):
        return "Federalism"
    else:
        return "Other"

cat2idx = {c: i for i, c in enumerate(CATEGORY_MAP)}
idx2cat = {i: c for c, i in cat2idx.items()}



Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Training GCN...
Training GraphSAGE...


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/m


GCN Test Metrics:
Accuracy: 0.7660, F1: 0.8016
Classification Report:
               precision    recall  f1-score   support

      Rights       1.00      0.88      0.93         8
  Governance       0.67      0.67      0.67         9
   Judiciary       0.00      0.00      0.00         0
  Federalism       0.89      0.80      0.84        10
       Other       0.83      0.75      0.79        20

    accuracy                           0.77        47
   macro avg       0.68      0.62      0.65        47
weighted avg       0.84      0.77      0.80        47

Confusion Matrix:
 [[ 7  0  1  0  0]
 [ 0  6  0  0  3]
 [ 0  0  0  0  0]
 [ 0  0  2  8  0]
 [ 0  3  1  1 15]]

GraphSAGE Test Metrics:
Accuracy: 0.7234, F1: 0.7440
Classification Report:
               precision    recall  f1-score   support

      Rights       1.00      0.88      0.93         8
  Governance       0.57      0.44      0.50         9
   Judiciary       0.00      0.00      0.00         0
  Federalism       0.80      0.80 

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.p

#  FETCH DATA FROM NEO4J


In [ ]:

def fetch_graph_from_neo4j(uri, user, password):
    driver = GraphDatabase.driver(uri, auth=(user, password))

    def fetch_nodes(tx):
        query = """
        MATCH (n:Article)
        RETURN id(n) AS neo4j_id,
               n.id AS article_id,
               n.text AS text,
               n.title AS title
        """
        return [r.data() for r in tx.run(query)]

    def fetch_edges(tx):
        query = """
        MATCH (a:Article)-[:RELATES_TO]->(b:Article)
        RETURN id(a) AS source_id, id(b) AS target_id
        """
        return [r.data() for r in tx.run(query)]

    with driver.session() as session:
        nodes = session.execute_read(fetch_nodes)
        edges = session.execute_read(fetch_edges)

    driver.close()
    return nodes, edges


# BUILD GRAPH

In [ ]:


def build_graph(nodes, edges, embed_model="all-mpnet-base-v2"):
    embedder = SentenceTransformer(embed_model)
    texts, ids, node_map = [], [], {}
    labels = {}

    for i, n in enumerate(nodes):
        aid = (n.get("article_id") or f"article:{i}").lower()
        text = n.get("text") or n.get("title") or ""
        node_map[n["neo4j_id"]] = i
        texts.append(text)
        ids.append(aid)
        labels[aid] = map_category(aid)  # programmatic mapping

    # Generate embeddings
    embeddings = embedder.encode(
        texts, batch_size=32, show_progress_bar=True, normalize_embeddings=True
    )

    data = HeteroData()
    data["Article"].x = torch.tensor(embeddings, dtype=torch.float)

    # Assign labels
    y = torch.full((len(ids),), -1, dtype=torch.long)
    for i, aid in enumerate(ids):
        if aid in labels:
            y[i] = cat2idx[labels[aid]]
    data["Article"].y = y

    # Build edges
    edge_list = []
    for e in edges:
        src = node_map.get(e["source_id"])
        tgt = node_map.get(e["target_id"])
        if src is not None and tgt is not None:
            edge_list.append([src, tgt])

    if edge_list:
        data["Article", "relates_to", "Article"].edge_index = torch.tensor(edge_list).t().contiguous()
        data = ToUndirected()(data)  # make bidirectional

    return data


# Train/Test/Val split

In [ ]:

# STEP 3: CREATE SPLITS
def create_splits(data, train_ratio=0.7, val_ratio=0.15):
    idx = (data["Article"].y != -1).nonzero(as_tuple=True)[0]
    perm = torch.randperm(len(idx))

    n_train = int(train_ratio * len(idx))
    n_val = int(val_ratio * len(idx))

    train_idx = idx[perm[:n_train]]
    val_idx = idx[perm[n_train:n_train + n_val]]
    test_idx = idx[perm[n_train + n_val:]]

    for mask in ["train_mask", "val_mask", "test_mask"]:
        data["Article"][mask] = torch.zeros(data["Article"].x.size(0), dtype=torch.bool)

    data["Article"].train_mask[train_idx] = True
    data["Article"].val_mask[val_idx] = True
    data["Article"].test_mask[test_idx] = True

    return data


# Models

In [ ]:
class GCN(torch.nn.Module):
    def __init__(self, in_c, hid_c, out_c):
        super().__init__()
        self.c1 = GCNConv(in_c, hid_c)
        self.c2 = GCNConv(hid_c, hid_c)
        self.c3 = GCNConv(hid_c, out_c)

    def forward(self, x, edge_index):
        x = F.relu(self.c1(x, edge_index))
        x = F.dropout(x, 0.3, training=self.training)
        x = F.relu(self.c2(x, edge_index))
        x = F.dropout(x, 0.3, training=self.training)
        return self.c3(x, edge_index)

class GraphSAGE(torch.nn.Module):
    def __init__(self, in_c, hid_c, out_c):
        super().__init__()
        self.c1 = SAGEConv(in_c, hid_c, aggr="mean")
        self.c2 = SAGEConv(hid_c, hid_c, aggr="mean")
        self.c3 = SAGEConv(hid_c, out_c, aggr="mean")

    def forward(self, x, edge_index):
        x = F.relu(self.c1(x, edge_index))
        x = F.dropout(x, 0.3, training=self.training)
        x = F.relu(self.c2(x, edge_index))
        x = F.dropout(x, 0.3, training=self.training)
        return self.c3(x, edge_index)


# TRAINING

In [ ]:
def train_epoch(model, data, optimizer):
    model.train()
    optimizer.zero_grad()
    out = model(data["Article"].x, data["Article", "relates_to", "Article"].edge_index)
    mask = data["Article"].train_mask
    labels = data["Article"].y[mask]

    counts = Counter(labels.tolist())
    weights = torch.tensor([1.0 / counts.get(i, 1) for i in range(out.size(1))], device=out.device)

    loss = F.cross_entropy(out[mask], labels, weight=weights)
    loss.backward()
    optimizer.step()
    return loss.item()

@torch.no_grad()
def evaluate(model, data, mask_name):
    model.eval()
    out = model(data["Article"].x, data["Article", "relates_to", "Article"].edge_index)
    pred = out.argmax(dim=1)
    mask = data["Article"][mask_name]
    y_true = data["Article"].y[mask].cpu().numpy()
    y_pred = pred[mask].cpu().numpy()

    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average="weighted")
    report = classification_report(y_true, y_pred, target_names=list(cat2idx.keys()))
    cm = confusion_matrix(y_true, y_pred)
    return acc, f1, report, cm

def train_model(model, data, epochs=200):
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.005, weight_decay=1e-4)
    best_f1, best_state = 0, None
    patience, wait = 20, 0

    for epoch in range(epochs):
        train_epoch(model, data, optimizer)
        acc, f1, _, _ = evaluate(model, data, "val_mask")

        if f1 > best_f1:
            best_f1 = f1
            best_state = model.state_dict()
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break

    model.load_state_dict(best_state)
    return model


In [ ]:

# STEP 6: SAVE MODEL

def save_model(model, name):
    os.makedirs("saved_models", exist_ok=True)
    torch.save(model.state_dict(), f"saved_models/{name}.pt")
    meta = {"cat2idx": cat2idx, "idx2cat": idx2cat}
    with open("saved_models/metadata.json", "w") as f:
        import json
        json.dump(meta, f, indent=2)


In [ ]:
# MAIN
def main():
    URI = "Replace your uri"
    USER = "neo4j"
    PASSWORD = "Password" # replace with your password

    # Fetch graph
    nodes, edges = fetch_graph_from_neo4j(URI, USER, PASSWORD)
    data = build_graph(nodes, edges)
    data = create_splits(data)

    in_c = data["Article"].x.size(1)
    out_c = len(cat2idx)

    # Train models
    print("Training GCN...")
    gcn_model = train_model(GCN(in_c, 256, out_c), data)
    print("Training GraphSAGE...")
    sage_model = train_model(GraphSAGE(in_c, 256, out_c), data)

    # Evaluate on test
    print("\nGCN Test Metrics:")
    acc, f1, report, cm = evaluate(gcn_model, data, "test_mask")
    print(f"Accuracy: {acc:.4f}, F1: {f1:.4f}")
    print("Classification Report:\n", report)
    print("Confusion Matrix:\n", cm)

    print("\nGraphSAGE Test Metrics:")
    acc, f1, report, cm = evaluate(sage_model, data, "test_mask")
    print(f"Accuracy: {acc:.4f}, F1: {f1:.4f}")
    print("Classification Report:\n", report)
    print("Confusion Matrix:\n", cm)

    # Save models
    save_model(gcn_model, "gcn_article_classifier")
    save_model(sage_model, "sage_article_classifier")

    print("\n✅ Training complete & models saved.")

if __name__ == "__main__":
    main()


In [ ]:
def fetch_graph_from_neo4j(uri, user, password):
    driver = GraphDatabase.driver(uri, auth=(user, password))

    def fetch_nodes(tx):
        query = """
        MATCH (n:Article)
        RETURN id(n) AS neo4j_id,
               n.id AS article_id,
               n.text AS text,
               n.title AS title
        """
        return [r.data() for r in tx.run(query)]

    def fetch_edges(tx):
        query = """
        MATCH (a:Article)-[:RELATES_TO]->(b:Article)
        RETURN id(a) AS source_id, id(b) AS target_id
        """
        return [r.data() for r in tx.run(query)]

    with driver.session() as session:
        nodes = session.execute_read(fetch_nodes)
        edges = session.execute_read(fetch_edges)

    driver.close()
    return nodes, edges

# -------------------------
# STEP 2: BUILD GRAPH
# -------------------------
def build_graph(nodes, edges, embed_model="all-mpnet-base-v2"):
    embedder = SentenceTransformer(embed_model)
    texts, ids, node_map = [], [], {}
    labels = {}

    for i, n in enumerate(nodes):
        aid = (n.get("article_id") or f"article:{i}").lower()
        text = n.get("text") or n.get("title") or ""
        node_map[n["neo4j_id"]] = i
        texts.append(text)
        ids.append(aid)
        labels[aid] = map_category(aid)  # programmatic mapping

    # Generate embeddings
    embeddings = embedder.encode(
        texts, batch_size=32, show_progress_bar=True, normalize_embeddings=True
    )

    data = HeteroData()
    data["Article"].x = torch.tensor(embeddings, dtype=torch.float)

    # Assign labels
    y = torch.full((len(ids),), -1, dtype=torch.long)
    for i, aid in enumerate(ids):
        if aid in labels:
            y[i] = cat2idx[labels[aid]]
    data["Article"].y = y

    # Build edges
    edge_list = []
    for e in edges:
        src = node_map.get(e["source_id"])
        tgt = node_map.get(e["target_id"])
        if src is not None and tgt is not None:
            edge_list.append([src, tgt])

    if edge_list:
        data["Article", "relates_to", "Article"].edge_index = torch.tensor(edge_list).t().contiguous()
        data = ToUndirected()(data)  # make bidirectional

    return data

# -------------------------
# STEP 3: CREATE SPLITS
# -------------------------
def create_splits(data, train_ratio=0.7, val_ratio=0.15):
    idx = (data["Article"].y != -1).nonzero(as_tuple=True)[0]
    perm = torch.randperm(len(idx))

    n_train = int(train_ratio * len(idx))
    n_val = int(val_ratio * len(idx))

    train_idx = idx[perm[:n_train]]
    val_idx = idx[perm[n_train:n_train + n_val]]
    test_idx = idx[perm[n_train + n_val:]]

    for mask in ["train_mask", "val_mask", "test_mask"]:
        data["Article"][mask] = torch.zeros(data["Article"].x.size(0), dtype=torch.bool)

    data["Article"].train_mask[train_idx] = True
    data["Article"].val_mask[val_idx] = True
    data["Article"].test_mask[test_idx] = True

    return data

# -------------------------
# STEP 4: MODELS
# -------------------------
class GCN(torch.nn.Module):
    def __init__(self, in_c, hid_c, out_c):
        super().__init__()
        self.c1 = GCNConv(in_c, hid_c)
        self.c2 = GCNConv(hid_c, hid_c)
        self.c3 = GCNConv(hid_c, out_c)

    def forward(self, x, edge_index):
        x = F.relu(self.c1(x, edge_index))
        x = F.dropout(x, 0.3, training=self.training)
        x = F.relu(self.c2(x, edge_index))
        x = F.dropout(x, 0.3, training=self.training)
        return self.c3(x, edge_index)

class GraphSAGE(torch.nn.Module):
    def __init__(self, in_c, hid_c, out_c):
        super().__init__()
        self.c1 = SAGEConv(in_c, hid_c, aggr="mean")
        self.c2 = SAGEConv(hid_c, hid_c, aggr="mean")
        self.c3 = SAGEConv(hid_c, out_c, aggr="mean")

    def forward(self, x, edge_index):
        x = F.relu(self.c1(x, edge_index))
        x = F.dropout(x, 0.3, training=self.training)
        x = F.relu(self.c2(x, edge_index))
        x = F.dropout(x, 0.3, training=self.training)
        return self.c3(x, edge_index)

# -------------------------
# STEP 5: TRAINING
# -------------------------
def train_epoch(model, data, optimizer):
    model.train()
    optimizer.zero_grad()
    out = model(data["Article"].x, data["Article", "relates_to", "Article"].edge_index)
    mask = data["Article"].train_mask
    labels = data["Article"].y[mask]

    counts = Counter(labels.tolist())
    weights = torch.tensor([1.0 / counts.get(i, 1) for i in range(out.size(1))], device=out.device)

    loss = F.cross_entropy(out[mask], labels, weight=weights)
    loss.backward()
    optimizer.step()
    return loss.item()

@torch.no_grad()
def evaluate(model, data, mask_name):
    model.eval()
    out = model(data["Article"].x, data["Article", "relates_to", "Article"].edge_index)
    pred = out.argmax(dim=1)
    mask = data["Article"][mask_name]
    y_true = data["Article"].y[mask].cpu().numpy()
    y_pred = pred[mask].cpu().numpy()

    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average="weighted")
    report = classification_report(y_true, y_pred, target_names=list(cat2idx.keys()))
    cm = confusion_matrix(y_true, y_pred)
    return acc, f1, report, cm

def train_model(model, data, epochs=200):
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.005, weight_decay=1e-4)
    best_f1, best_state = 0, None
    patience, wait = 20, 0

    for epoch in range(epochs):
        train_epoch(model, data, optimizer)
        acc, f1, _, _ = evaluate(model, data, "val_mask")

        if f1 > best_f1:
            best_f1 = f1
            best_state = model.state_dict()
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break

    model.load_state_dict(best_state)
    return model

# -------------------------
# STEP 6: SAVE MODEL
# -------------------------
def save_model(model, name):
    os.makedirs("saved_models", exist_ok=True)
    torch.save(model.state_dict(), f"saved_models/{name}.pt")
    meta = {"cat2idx": cat2idx, "idx2cat": idx2cat}
    with open("saved_models/metadata.json", "w") as f:
        import json
        json.dump(meta, f, indent=2)

# -------------------------
# MAIN
# -------------------------
def main():
    URI = "neo4j+s://95909642.databases.neo4j.io"
    USER = "neo4j"
    PASSWORD = "dZh58cNxycNT5zgDxhVTYzgQTzc54UbaKJHEHC42VLY" # replace with your password

    # Fetch graph
    nodes, edges = fetch_graph_from_neo4j(URI, USER, PASSWORD)
    data = build_graph(nodes, edges)
    data = create_splits(data)

    in_c = data["Article"].x.size(1)
    out_c = len(cat2idx)

    # Train models
    print("Training GCN...")
    gcn_model = train_model(GCN(in_c, 256, out_c), data)
    print("Training GraphSAGE...")
    sage_model = train_model(GraphSAGE(in_c, 256, out_c), data)

    # Evaluate on test
    print("\nGCN Test Metrics:")
    acc, f1, report, cm = evaluate(gcn_model, data, "test_mask")
    print(f"Accuracy: {acc:.4f}, F1: {f1:.4f}")
    print("Classification Report:\n", report)
    print("Confusion Matrix:\n", cm)

    print("\nGraphSAGE Test Metrics:")
    acc, f1, report, cm = evaluate(sage_model, data, "test_mask")
    print(f"Accuracy: {acc:.4f}, F1: {f1:.4f}")
    print("Classification Report:\n", report)
    print("Confusion Matrix:\n", cm)

    # Save models
    save_model(gcn_model, "gcn_article_classifier")
    save_model(sage_model, "sage_article_classifier")

    print("\n✅ Training complete & models saved.")

if __name__ == "__main__":
    main()
